# Architecture Walkthrough — `src/02_transformer_ehr.py`

A step-by-step companion to the model in `src/02_transformer_ehr.py`. Every shape, default and
mask behaviour below is **measured on this machine**, not asserted. Cells that print a number
are there so you can check the claim yourself.

**How to use:** run cells top to bottom. Cells 1–10 are seconds-fast and involve no training.
Cell 11 is the only slow one and is fenced off deliberately.

| Section | Cells | What it establishes |
|---|---|---|
| Setup | 1 | environment, versions |
| The data | 2–3 | one **visit** is one token |
| The model | 4–6 | verified defaults, shape trace, parameter budget |
| Masks | 7 | why masked pooling is load-bearing |
| Attention | 8–9 | real weights, and Q/K/V by hand |
| Learning | 10–11 | one training step; optional short run |
| Play | 12 | sandbox |

## 1. Environment

Nothing here needs `scikit-learn` — the tensor mechanics are pure `torch`. The optional training
cell at the end does use it for AUROC/AUPRC, exactly as `src/02_transformer_ehr.py` does.

In [ ]:
import sys, math, platform
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "sequences.npz").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from attention_utils import extract_attention, build_ehr_transformer, count_parameters

print("python      ", platform.python_version())
print("torch       ", torch.__version__)
print("numpy       ", np.__version__)
print("working dir ", ROOT)
print("cuda        ", torch.cuda.is_available())

# Defaults are worth reading off the installed version rather than trusting docs.
L = nn.TransformerEncoderLayer(d_model=32, nhead=4, dim_feedforward=64, batch_first=True)
print("\nnn.TransformerEncoderLayer defaults actually in force here:")
print("  norm_first =", L.norm_first, "->", "POST-LN (original 2017 arrangement)" if not L.norm_first else "pre-LN")
print("  dropout    =", L.dropout)
print("  eps        =", L.norm1.eps)
act = L.activation
if act is nn.ReLU or act == nn.ReLU:
    act_name = "ReLU (nn.ReLU module)"
elif getattr(act, "__name__", "") == "relu":
    act_name = "ReLU (function: torch.nn.functional.relu)"
else:
    act_name = act
print("  activation =", act_name)
print("  child order:", [n for n, _ in L.named_children()])

## 2. The data — what one "token" actually is

The single most important idea: **a sequence element here is a clinic visit, not a word.**

Each patient is one sequence. Each visit is one element of that sequence. A visit is a row of six
numbers, generated in `src/generate_synthetic_ehr.py:33`:

```python
visits.append([hr/100, sbp/200, wbc/15, diag/4, age/100, sex])
```

Patients have different visit counts, so sequences are **padded** to a fixed length and a **mask**
records which slots are real (`1`) and which are filler (`0`) — `generate_synthetic_ehr.py:57-61`.

In [ ]:
blob = np.load(ROOT / "data" / "sequences.npz")
seqs, masks, labels = blob["seqs"], blob["masks"], blob["labels"]

print("seqs ", seqs.shape, seqs.dtype, " (patient, visit, feature)")
print("masks", masks.shape, masks.dtype, " 1 = real visit, 0 = padding")
print("labels", labels.shape, labels.dtype, f" prevalence {labels.mean():.4f}")
print("NaN in seqs:", bool(np.isnan(seqs).any()), "- imputation already happened in the generator")

print("\nfeature ranges (note the 0.000 floor is the padding, not the data):")
for i, name in enumerate(["hr/100", "sbp/200", "wbc/15", "diag/4", "age/100", "sex"]):
    print(f"  {name:9s} min {seqs[..., i].min():+.3f}   max {seqs[..., i].max():+.3f}")

print("\nvisits per patient: min", int(masks.sum(1).min()), " max", int(masks.sum(1).max()))

# Confirm every mask is a clean run of 1s followed by 0s - i.e. padding is always at the end.
prefix = np.zeros_like(masks)
for r in range(masks.shape[0]):
    prefix[r, : int(masks[r].sum())] = 1
print("every mask is 1s-then-0s:", bool((masks == prefix).all()))

### Padding collides with real values

Look at the table above: `diag/4` and `sex` are genuinely `0.0` for some real visits. **A padded
slot is also `0.0`.** Nothing in the numbers alone distinguishes "this patient has no diagnosis
coded" from "this visit does not exist". That ambiguity is the entire reason the mask exists.

In [ ]:
# Pick a positive patient and print the grid exactly as the model receives it.
i = int(np.where(labels == 1)[0][0])
nv = int(masks[i].sum())

print(f"patient #{i}: {nv} real visits, label {labels[i]}")
print(f"\nseqs[{i}]  shape {seqs[i].shape}   (T={seqs[i].shape[0]} slots x 6 features)")
print(" " * 10 + "".join(f"{n:>10s}" for n in ["hr/100", "sbp/200", "wbc/15", "diag/4", "age/100", "sex"]))
for t in range(seqs[i].shape[0]):
    tag = "" if masks[i, t] else "   <- PADDING"
    print(f"  visit {t} " + "".join(f"{v:10.3f}" for v in seqs[i, t]) + tag)

print(f"\nmasks[{i}] = {masks[i].astype(int)}")

# A tensor this is exactly what the model consumes: floats for features and mask alike.
x = torch.tensor(seqs, dtype=torch.float32)
m = torch.tensor(masks, dtype=torch.float32)
print("\nmodel input x ", tuple(x.shape), x.dtype)
print("model mask  m ", tuple(m.shape), m.dtype)
print("one patient slice x[i] ", tuple(x[i].shape), "-> (T, d_in): one row per visit")

## 3. The model

`EHRTransformer` (`src/02_transformer_ehr.py:52-70`) in full — this is the whole architecture:

```python
self.input_proj = torch.nn.Linear(d_in, d_model)                    # 6  -> 32
self.pos_emb    = torch.nn.Parameter(torch.randn(1, 8, d_model)*0.1)
enc_layer       = torch.nn.TransformerEncoderLayer(
                      d_model=d_model, nhead=nhead, dim_feedforward=64, batch_first=True)
self.encoder     = torch.nn.TransformerEncoder(enc_layer, num_layers=2,
                                               enable_nested_tensor=False)
self.classifier  = torch.nn.Linear(d_model, 1)

def forward(self, x, mask):
    h = self.input_proj(x) + self.pos_emb[:, :x.size(1), :]          # (B,8,6) -> (B,8,32)
    pad_mask = (mask == 0)                                          # True = ignore
    h = self.encoder(h, src_key_padding_mask=pad_mask)              # (B,8,32)
    mask_e = mask.unsqueeze(-1)
    pooled = (h * mask_e).sum(1) / mask_e.sum(1).clamp(min=1)       # (B,8,32) -> (B,32)
    return self.classifier(pooled).squeeze(-1)                      # (B,32) -> (B,)
```

`build_ehr_transformer()` in `attention_utils` constructs exactly this, so the notebook and the
script cannot drift apart. Cell 6 proves they match.

In [ ]:
model = build_ehr_transformer()
print(type(model).__name__)
print("\nsubmodules:")
for name, mod in model.named_children():
    print(f"  {name:12s} {type(mod).__name__}")

print("\nlayer count:", len(model.encoder.layers))
print("layers are independent objects:", model.encoder.layers[0] is not model.encoder.layers[1])
same = all(
    torch.equal(a, b) for a, b in zip(model.encoder.layers[0].parameters(),
                                      model.encoder.layers[1].parameters())
)
print("layers start with identical values:", same, "(deep-copied, NOT weight-tied)")

L0 = model.encoder.layers[0]
print("\ninside one encoder layer:")
print("  self_attn     ", type(L0.self_attn).__name__, "heads =", L0.self_attn.num_heads)
print("  feedforward   ", L0.linear1.in_features, "->", L0.linear1.out_features, "->", L0.linear2.out_features, "  (dim_feedforward=64 is the middle)")
print("  final norm on the stack:", model.encoder.norm, "(None = no extra norm after the last block)")

## 4. The forward pass, one stage at a time

Four axes matter, and confusing them is the usual source of confusion:

| axis | size | meaning |
|---|---|---|
| `B` | 64 | patients in the batch |
| `T` | 8 | visit slots (padded) |
| `d_in` | 6 | features per visit |
| `d_model` | 32 | hidden coordinates per visit |

Attention operates over **`T`** — visits attend to visits. The 6 features are mixed only *inside*
the Q/K/V projection, never against each other by attention.

In [ ]:
B = 4
xb   = x[:B]
mb   = m[:B]

stages = {}
stages["input x"] = tuple(xb.shape)

h1 = model.input_proj(xb)
stages["after input_proj (nn.Linear 6->32)"] = tuple(h1.shape)

h2 = h1 + model.pos_emb[:, : xb.size(1), :]
stages["after + pos_emb"] = tuple(h2.shape)

with torch.no_grad():
    h3 = model.encoder(h2, src_key_padding_mask=(mb == 0))
stages["after encoder (2 layers, 4 heads)"] = tuple(h3.shape)

mask_e = mb.unsqueeze(-1)
pooled = (h3 * mask_e).sum(1) / mask_e.sum(1).clamp(min=1)
stages["after masked mean pooling"] = tuple(pooled.shape)

with torch.no_grad():
    logits = model(xb, mb)
stages["after classifier (nn.Linear 32->1)"] = tuple(logits.shape)

for k, v in stages.items():
    print(f"  {k:38s} {v}")

print(f"\naxes: B={B} patients, T={xb.size(1)} visit slots, d_in=6 features, d_model=32 hidden")
print("one logit per patient:", tuple(logits.shape), "-> squeeze(-1) removed the trailing 1")

## 5. What replaced tokenization and the embedding table?

In a word-based model each token is an integer id and an embedding table maps id → vector. Here a
visit arrives **already as six numbers**, so there is nothing to look up: projecting those six
numbers *is* the embedding step.

The consequence is worth stating plainly: `nn.Embedding` appears **zero times** anywhere in this
repository's `src/`. Check it:

```python
# there is no vocabulary because there is nothing to discretise
```

What remains of the idea is the **positional** table — the same mechanism as a learned position
embedding, just bounded to 8 slots because a patient has at most 8 visits in this dataset.

In [ ]:
# Demonstrate the difference concretely: a lookup vs a projection.
vocab = {("visit 0",): 0, ("visit 1",): 1}
tbl = nn.Embedding(2, 32)
print("nn.Embedding(2, 32): row lookup by integer id ->", tuple(tbl.weight[1].shape))

print("nn.Linear(6, 32):    projection of a feature vector ->",
      tuple(model.input_proj.weight.shape), "(no id exists to look up)")

print("\nembedding layers anywhere in src/:",
      sum("nn.Embedding" in Path(p).read_text()
          for p in (ROOT / "src").glob("*.py")), "(zero, by design)")

# The positional table is the surviving lookup.
print("\npos_emb table:", tuple(model.pos_emb.shape), "= (1, max_visits=8, d_model=32)")
print("sliced per batch:", tuple(model.pos_emb[:, : xb.size(1), :].shape))
print("\ncosine similarity between position vectors (are they distinguishable?):")
pe = model.pos_emb[0].detach()
pe_n = pe / pe.norm(dim=-1, keepdim=True)
cos = (pe_n @ pe_n.T).numpy()
print(np.round(cos, 2))

## 6. Parameter budget

Where the 17,601 parameters actually live. Almost everything is in the encoder — the input
projection and classifier are rounding errors by comparison.

In [ ]:
total = count_parameters(model)
print(f"total parameters: {total:,}\n")

groups = {
    "input_proj (Linear 6->32)": model.input_proj,
    "encoder (2 layers)": model.encoder,
    "classifier (Linear 32->1)": model.classifier,
}
for name, mod in groups.items():
    n = sum(p.numel() for p in mod.parameters())
    print(f"  {name:30s} {n:7,}  ({100*n/total:5.1f}%)")
print(f"  {'pos_emb (Parameter)':30s} {model.pos_emb.numel():7,}  ({100*model.pos_emb.numel()/total:5.1f}%)")

print("\nper encoder layer:")
for n, p in model.encoder.layers[0].named_parameters():
    print(f"  {n:32s} {tuple(p.shape)}")

## 7. Masks — the subtle part

Three different things get called "masking". Only one is used here.

| kind | what it does | present here? |
|---|---|---|
| **padding mask** | hides filler slots so they cannot be attended to | **yes** — `src_key_padding_mask` |
| **causal mask** | hides *future* positions, for left-to-right generation | **no** |
| **missing-feature mask** | flags an absent *measurement* within a visit | **no** — the generator imputes NaN → 0.7 at `generate_synthetic_ehr.py:41` |

There is no causal mask because there is no generation loop: the label is computed from the
*complete* record, so hiding later visits from earlier ones would only throw information away.

The padding mask is **one-directional**, and this is the part that surprises people:

- a padded **key** column is exactly `0` — nobody attends to a filler slot;
- a padded **query** row is **not** zero — a filler slot still computes a full attention
  distribution over the real visits and produces a genuine output vector.

So a padded slot manufactures a plausible-looking summary of the patient. `masked mean pooling`
(`src/02_transformer_ehr.py:68-69`) is what stops those fabrications reaching the prediction.

In [ ]:
# One patient, 5 real visits, 3 padding slots.
demo_x = torch.tensor(seqs[i:i+1, :8], dtype=torch.float32)
demo_m = torch.tensor(masks[i:i+1, :8], dtype=torch.float32)
demo_nv = int(demo_m.sum())

with torch.no_grad():
    attn = extract_attention(model, demo_x, demo_m)
w = attn[0][0].numpy()          # (nhead, T, T)
print(f"attention tensors: {len(attn)} layers x {w.shape[0]} heads x {w.shape[1]} queries x {w.shape[2]} keys")
print(f"patient has {demo_nv} real visits, {demo_m.shape[1]-demo_nv} padded slots\n")

for head in range(w.shape[0]):
    mat = w[head]
    print(f"  layer 0, head {head}")
    for q in range(mat.shape[0]):
        tag = "  <- padded query" if q >= demo_nv else ""
        print(f"    q{q}: " + " ".join(f"{v:.3f}" for v in mat[q]) + tag)

print()
for head in range(w.shape[0]):
    mat = w[head]
    print(f"  head {head}: valid query row sums {[round(float(mat[q].sum()),4) for q in range(demo_nv)]}"
          f"  padded-query row sum {mat[demo_nv].sum():.4f}"
          f"  padded-key col max {np.abs(mat[:, demo_nv:]).max():.6f}")

In [ ]:
fig, axes = plt.subplots(1, w.shape[0], figsize=(3.1 * w.shape[0], 3.4))
for head in range(w.shape[0]):
    ax = axes[head]
    im = ax.imshow(w[head], cmap="viridis", vmin=0, vmax=1)
    ax.axhline(demo_nv - 0.5, color="white", lw=1.5, ls="--")
    ax.axvline(demo_nv - 0.5, color="white", lw=1.5, ls="--")
    ax.set_title(f"head {head}", fontsize=10)
    ax.set_xlabel("key", fontsize=8); ax.set_ylabel("query", fontsize=8)
    ax.tick_params(labelsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"Patient #{i}: {demo_nv} real visits. Dashed line = padding boundary.\n"
             "Right of the line: exactly 0 (keys ignored).  Below the line: NOT zero (queries still active).",
             fontsize=9)
fig.tight_layout(); plt.show()

In [ ]:
# Prove the pooling is what removes the padded fabrications.
with torch.no_grad():
    h = model.encoder(model.input_proj(demo_x) + model.pos_emb[:, :8, :],
                      src_key_padding_mask=(demo_m == 0))

masked_pool = (h * demo_m.unsqueeze(-1)).sum(1) / demo_m.unsqueeze(-1).sum(1).clamp(min=1)
naive_pool  = h.mean(dim=1)

print("per-visit output norms — padded slots are NOT zero, they are real vectors:")
print("  norms:", np.round(h[0].norm(dim=-1).numpy(), 3))
print("\nso an unmasked mean would fold in", 8 - demo_nv, "fabricated summaries:")
print("  masked mean pooling ->", np.round(masked_pool[0].numpy()[:4], 4))
print("  plain .mean(1)      ->", np.round(naive_pool[0].numpy()[:4], 4))
print("  max abs difference  ->", float((masked_pool - naive_pool).abs().max()))

## 8. Real attention weights

Cell 7 already called `extract_attention`. It is worth knowing **why** that function exists, because
the obvious approach does not work:

- `nn.TransformerEncoderLayer` calls attention internally with `need_weights=False`;
- on torch 2.14 `nn.TransformerEncoder.forward()` does not accept `need_weights` at all.

```python
model.encoder(h, need_weights=True)
# TypeError: TransformerEncoder.forward() got an unexpected keyword argument 'need_weights'
```

So `extract_attention` hooks each `self_attn` module to capture the tensor attention actually
receives, then replays it with `need_weights=True`. Below is the same thing done by hand, from
`in_proj_weight` — which is also the clearest way to see where Q, K and V come from.

In [ ]:
layer = model.encoder.layers[0]
mha = layer.self_attn
D, H = 32, mha.num_heads
d_head = D // H
print(f"d_model {D}, heads {H}, head_dim {d_head}")

# in_proj_weight does all three projections at once: q, k, v stacked -> (3*d_model, d_model)
print("in_proj_weight", tuple(mha.in_proj_weight.shape), "= (3*D, D)  -> split into Q, K, V")
print("out_proj      ", tuple(mha.out_proj.weight.shape), "= (D, D)     -> recombines heads")

# Recompute the attention for patient #i straight from the projection weights.
captured = {}
handle = mha.register_forward_pre_hook(lambda mod, a: captured.__setitem__("h", a[0].detach()))
with torch.no_grad():
    model(demo_x, demo_m)
handle.remove()
h = captured["h"]

with torch.no_grad():
    qkv = h @ mha.in_proj_weight.T + mha.in_proj_bias
    q, k, v = qkv.split(D, dim=-1)
    q = q.view(1, 8, H, d_head).transpose(1, 2)
    k = k.view(1, 8, H, d_head).transpose(1, 2)
    v = v.view(1, 8, H, d_head).transpose(1, 2)
    print(f"\nQ {tuple(q.shape)}  K {tuple(k.shape)}  V {tuple(v.shape)}  (B, H, T, head_dim)")

    scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)
    scores = scores.masked_fill((demo_m == 0)[:, None, None, :], float("-inf"))
    manual = scores.softmax(-1)

print(f"\nhand-computed softmax(QK^T/sqrt({d_head}) + padding mask): {tuple(manual.shape)}")
print("valid query rows sum to 1 (per head):",
      [[round(float(manual[0, hh, qq].sum()), 6) for qq in range(demo_nv)] for hh in range(H)])
print("padded key columns max   :", float(manual[0, :, :, demo_nv:].abs().max()))
print("padded query row sum     :", round(float(manual[0, 0, demo_nv].sum()), 6))

got = attn[0]
print("\nextracted by attention_utils vs hand-computed: max abs diff =",
      f"{float((manual - got).abs().max()):.2e}")
print("=> extract_attention is exactly softmax(QK^T/sqrt(head_dim) + padding mask)")

## 9. What the trained model actually learned

Load the checkpoint and look at where attention lands. This is the same view `app.py` serves
interactively.

The checkpoint behind it was selected by **validation** AUROC on a stratified 70/15/15 split, and
the script is fully seeded, so two consecutive runs produce a byte-identical file. The test set is
never consulted during training.

In [ ]:
ckpt = ROOT / "results" / "best_transformer.pt"
if ckpt.exists():
    trained = build_ehr_transformer()
    trained.load_state_dict(torch.load(ckpt, map_location="cpu"))
    trained.eval()
    with torch.no_grad():
        learned = extract_attention(trained, x[:256], m[:256])
    w = learned[1][i].numpy()   # layer 1, patient #i
    print("attention received per visit (mean over heads, layer 1):")
    mass = w[:, :demo_nv, :demo_nv].mean(axis=(0, 1))
    for t in range(demo_nv):
        bar = "#" * int(round(mass[t] * 60))
        print(f"  visit {t}: {mass[t]:.3f} {bar}")
    print(f"\n  padding columns: {np.abs(w[:, :, demo_nv:]).max():.6f} (exactly zero)")
    best_layer1_head = int(np.argmax(w[:, :demo_nv, :demo_nv].mean(axis=(1, 2))))
    print(f"  most-attended-to visit: {int(np.argmax(mass))}   "
          f"most self-focused head: {best_layer1_head}")
else:
    print("no checkpoint — run: python src/02_transformer_ehr.py  (~10 min on CPU)")

### Did the heads find the label?

The V1 label is a *trend*: `wbc[-1] - wbc[0]` (`src/generate_synthetic_ehr.py:42`) — a difference
between the **first** and the **last** visit. Nothing tells the model that visit 0 and visit 7 are
special.

So it is worth asking whether some head ended up reading one end of the record. Below, each head's
share of attention mass on the first and last real visit is averaged over 1,000 patients. Chance
level is roughly `1/n_visits`.


In [ ]:
if ckpt.exists():
    N = 1000
    nv_all = m[:N].sum(1).numpy().astype(int)

    with torch.no_grad():
        all_attn = extract_attention(trained, x[:N], m[:N])

    for layer_i in (0, 1):
        wn = all_attn[layer_i].numpy()          # (N, nhead, T, T)
        first = np.zeros(wn.shape[1])
        last = np.zeros(wn.shape[1])
        chance = 0.0
        for b in range(N):
            k = int(nv_all[b])
            first += wn[b][:, :, 0].mean(axis=1)      # mean over query rows
            last += wn[b][:, :, k - 1].mean(axis=1)
            chance += 1.0 / k
        first /= N; last /= N; chance /= N
        print(f"layer {layer_i}   chance level ~ {chance:.3f}")
        for h in range(wn.shape[1]):
            ratio = last[h] / first[h]
            role = "recency head" if ratio > 1.5 else ("baseline head" if ratio < 0.7 else "mixed")
            print(f"  head {h}: first {first[h]:.3f}  last {last[h]:.3f}  "
                  f"last/first {ratio:.2f}  -> {role}")
        print()

    print("Layer 1 should show one head biased to each end. That is the label rule, recovered")
    print("by attention without being told about it. See docs/architecture.html section 6.")
else:
    print("no checkpoint - run: python src/02_transformer_ehr.py")


## 10. One training step

Prediction → loss → backward → update, with nothing hidden. The script's setup is at
`src/02_transformer_ehr.py:88-126`.

- **logit** — a single raw number per patient. `BCEWithLogitsLoss` applies the sigmoid internally,
  so there is no separate sigmoid step and the maths stays numerically stable.
- **`pos_weight`** — prevalence is 0.0977, so negatives outnumber positives roughly 9:1.
  `pos_weight` multiplies the positive term, so a model cannot win by predicting "no" every time.
  This cell derives it from its own 64-patient batch (7 positives -> 8.143). The script computes it
  on the stratified training split instead (`src/02_transformer_ehr.py:90` -> 9.24) and section 11's
  80/20 split gives 9.26 — the three differ only in which patients are counted.

In [ ]:
fresh = build_ehr_transformer()

batch_idx = np.arange(64)
xb_t = torch.tensor(seqs[batch_idx], dtype=torch.float32)
mb_t = torch.tensor(masks[batch_idx], dtype=torch.float32)
yb_t = torch.tensor(labels[batch_idx], dtype=torch.float32)

n_pos = float(yb_t.sum())
pos_weight = torch.tensor([(len(yb_t) - yb_t.sum()) / yb_t.sum()])
print(f"batch positives {n_pos:.0f}/{len(yb_t)}  ->  pos_weight = {float(pos_weight):.3f}")
print("pos_weight dtype:", pos_weight.dtype, "(matches the logits, so BCEWithLogitsLoss takes it directly)")

logits = fresh(xb_t, mb_t)
print(f"\nlogits {tuple(logits.shape)} -> sigmoid range "
      f"[{torch.sigmoid(logits).min():.4f}, {torch.sigmoid(logits).max():.4f}]")

crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
loss = crit(logits, yb_t)
print(f"loss (untrained model) = {float(loss):.4f}")

opt = torch.optim.AdamW(fresh.parameters(), lr=2e-3, weight_decay=1e-4)
opt.zero_grad()
loss.backward()

print("\ngradient norm per parameter group (all non-zero => everything is being learned):")
for name, p in fresh.named_parameters():
    if p.grad is not None:
        print(f"  {name:34s} {p.grad.norm().item():.3e}")
frozen = [n for n, p in fresh.named_parameters() if p.grad is None]
print("\nfrozen parameters:", frozen or "none — every parameter receives gradient")

before = fresh.classifier.weight.detach().clone()
opt.step()
print("classifier weight changed by step:", bool((fresh.classifier.weight.detach() - before).abs().max() > 0))

fresh.eval()
with torch.no_grad():
    print("eval mode -> dropout off, predictions deterministic:",
          bool(torch.equal(fresh(xb_t, mb_t), fresh(xb_t, mb_t))))

## 11. OPTIONAL — short training run

Slow. Runs a handful of epochs rather than the script's 35, then reports AUROC/AUPRC and ablates
the positional embedding. Skip this cell unless you want to watch a loss curve.

Uses `scikit-learn` for the metrics, exactly as `src/02_transformer_ehr.py:133-138` does.

In [ ]:
SLOW = True   # set False to skip

if SLOW:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, average_precision_score

    torch.manual_seed(0)
    Xtr, Xte, mtr, mte, ytr, yte = train_test_split(
        seqs, masks, labels, test_size=0.2, stratify=labels, random_state=42)
    xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    mtr_t = torch.tensor(mtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.float32)

    net = build_ehr_transformer()
    pw = torch.tensor([(len(ytr_t) - ytr_t.sum()) / ytr_t.sum()], dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimiser = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)

    ds = torch.utils.data.TensorDataset(xtr_t, mtr_t, ytr_t)
    loader = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=True)
    xt = torch.tensor(Xte, dtype=torch.float32)
    mt = torch.tensor(mte, dtype=torch.float32)

    history = []
    for epoch in range(6):
        net.train()
        for bx, bm, by in loader:
            optimiser.zero_grad()
            criterion(net(bx, bm), by).backward()
            optimiser.step()
        net.eval()
        with torch.no_grad():
            p = torch.sigmoid(net(xt, mt)).numpy()
        history.append((roc_auc_score(yte, p), average_precision_score(yte, p)))
        print(f"  epoch {epoch+1}: AUROC {history[-1][0]:.3f}  AUPRC {history[-1][1]:.3f}")

    print(f"\nbaseline AUPRC = prevalence = {labels.mean():.3f} (predicting 'no' every time)")

    # Ablation: zero the positional embedding and re-evaluate the trained net.
    with torch.no_grad():
        saved = net.pos_emb.detach().clone()
        net.pos_emb.zero_()
        p0 = torch.sigmoid(net(xt, mt)).numpy()
        net.pos_emb.copy_(saved)
    print(f"with pos_emb zeroed: AUROC {roc_auc_score(yte, p0):.3f}  "
          f"AUPRC {average_precision_score(yte, p0):.3f}")

    plt.plot([h[0] for h in history], marker="o", label="AUROC")
    plt.plot([h[1] for h in history], marker="s", label="AUPRC")
    plt.xlabel("epoch"); plt.ylabel("score"); plt.ylim(0, 1)
    plt.title(f"6 epochs only - full run is 35 (AUPRC floor = {labels.mean():.3f})")
    plt.legend(); plt.grid(alpha=0.3); plt.show()
else:
    print("skipped")

## 12. Sandbox

Change anything and re-run. A few starting points:

- set `d_model` to 64 in `build_ehr_transformer(d_model=64)` and watch the parameter count move
- pass `mask` with **all** zeros to see what `clamp(min=1)` in the pooling is protecting against
- compare a causal mask against the padding mask by adding `mask=` to the `encoder` call
- plot `w[:, :nv, :].sum(axis=(0,1))` for every patient to see how recency is weighted overall

In [ ]:
# 1. what happens with zero real visits?
empty_mask = torch.zeros(1, 8)
empty_x = torch.zeros(1, 8, 6)
try:
    out = model(empty_x, empty_mask)
    print("zero-real-visit batch ->", out.detach().numpy())

    # clamp(min=1) turns the would-be 0/0 into 0/1, so pooling returns zeros rather than NaN,
    # and the model output collapses to just classifier.bias - the prior for a patient with no visits.
    h_test = model.encoder(model.input_proj(empty_x) + model.pos_emb[:, :8, :],
                           src_key_padding_mask=(empty_mask == 0))
    me_t = empty_mask.unsqueeze(-1)
    noclamp = (h_test * me_t).sum(1) / me_t.sum(1)
    print("pooling WITHOUT clamp ->", noclamp.detach().numpy()[0, :4], "... (NaN)")
    print("clamp(min=1): denominator 0 -> 0/1 -> zero vector")
    print("output equals classifier.bias?", bool(torch.allclose(out, model.classifier.bias.detach())))

    # The safety is MODE-DEPENDENT. In eval mode the attention kernel returns NaN for an
    # all-masked row BEFORE pooling runs, so clamp cannot rescue it. No real patient reaches
    # either path: generate_synthetic_ehr.py:20 forces 2-8 visits.
    was_training = model.training
    model.eval()
    with torch.no_grad():
        eval_out = model(empty_x, empty_mask)
    model.train(was_training)
    print("same input, eval mode ->", eval_out.detach().numpy(), " <- NaN from attention, not pooling")
except Exception as e:
    print("raised:", type(e).__name__, e)

# 2. attention received per visit, averaged over many patients
if ckpt.exists():
    with torch.no_grad():
        a = extract_attention(trained, x[:256], m[:256])
    counts = m[:256].sum(1).long()
    for layer_i, w_layer in enumerate(a):
        wn = w_layer.numpy()
        totals = np.zeros(8)
        n = np.zeros(8)
        for b in range(len(wn)):
            nv_b = counts[b].item()
            totals[:nv_b] += wn[b][:, :nv_b, :nv_b].mean(axis=(0, 1))
            n[:nv_b] += 1
        profile = totals / np.maximum(n, 1)
        print(f"\nlayer {layer_i}: mean attention received by visit position (0=first visit)")
        print("  " + " ".join(f"{v:.3f}" if c else "  -  " for v, c in zip(profile, n > 0)))
else:
    print("\n(no checkpoint, skipping the aggregate profile)")